# Sparse Autoencoders Monosemantic Feature Discovery
## Learning Interpretable Representations from Overcomplete Dictionaries


### Resources used
- Olshausen & Field (1997) 'Sparse coding with an overcomplete basis set'. https://doi.org/10.1016/S0042-6989(97)00169-7
- Bricken et al. (2023) 'Towards Monosemanticity: Decomposing Language Models With Dictionary Learning'. https://transformer-circuits.pub/2023/monosemantic-features
- Cunningham et al. (2023) 'Sparse Autoencoders Find Highly Interpretable Features in Language Models'. https://arxiv.org/abs/2309.08600
- Elhage et al. (2022) 'Toy Models of Superposition'. https://transformer-circuits.pub/2022/toy_model
- Makhzani & Frey (2013) 'k-Sparse Autoencoders'. https://arxiv.org/abs/1312.5663
- Lee et al. (2006) 'Efficient sparse coding algorithms'. https://arxiv.org/abs/cs/0608094

### Accessibility
All figures use colourblind-safe palettes. Heatmaps use magma/RdYlGn colourmaps. Hatch patterns on bar charts. Alt-text in each Markdown cell.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'torch', 'matplotlib', 'numpy', 'scipy', '--quiet'])

In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.ndimage import uniform_filter1d
import warnings; warnings.filterwarnings('ignore')

# Dark amber/purple palette
BG   = '#0C0A14'; CARD = '#13101F'; GRID = '#1E1A30'
PURP = '#A78BFA'; AMBR = '#F59E0B'; ROSE = '#F472B6'
LIME = '#86EFAC'; CYAN = '#67E8F9'; TEXT = '#F1F5F9'; MUTE = '#94A3B8'

plt.rcParams.update({'figure.facecolor':BG,'axes.facecolor':CARD,
                     'font.family':'monospace','font.size':10})

def sa(ax):
    ax.set_facecolor(CARD); ax.tick_params(colors=MUTE,labelsize=9)
    ax.xaxis.label.set_color(TEXT); ax.yaxis.label.set_color(TEXT); ax.title.set_color(TEXT)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID)
    ax.grid(True,color=GRID,linewidth=0.7,alpha=0.8)

torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

---
## Section 1: The Superposition Problem

Why do neurons in trained neural networks respond to multiple unrelated concepts? This is the **superposition hypothesis** (Elhage et al., 2022): a network with d neurons can represent more than d features by encoding them in superposition as near-orthogonal directions in activation space.

**Alt-text:** Code block demonstrating the superposition problem numerically showing that d neurons can approximately represent n >> d features if the features are sparse enough.

In [ ]:
# Numerical demonstration of superposition
# A network with d neurons can represent n >> d features if features are sparse
# Key insight: if features rarely co-activate, near-orthogonal encoding works

d = 2   # network has 2 neurons
n = 5   # but we want to represent 5 features

# Create near-orthogonal 2D directions for 5 features
angles = np.linspace(0, np.pi, n, endpoint=False)  # Pentagon vertices
features_2d = np.column_stack([np.cos(angles), np.sin(angles)])  # (n, d)

# Simulate sparse feature activations: usually 0, occasionally 1
np.random.seed(42)
sparse_codes = np.random.binomial(1, 0.1, size=(1000, n)).astype(float)

# Encode to 2D neuron space
neuron_activations = sparse_codes @ features_2d  # (1000, d)

# Recovery: find which features are active given neuron activations
recovered = neuron_activations @ features_2d.T  # (1000, n) dot product with each feature
recovered_binary = (recovered > 0.5).astype(float)

# Accuracy: how often do we correctly recover the active features?
accuracy = (recovered_binary == sparse_codes).mean()
print(f'Superposition recovery accuracy: {accuracy*100:.1f}%')
print(f'd={d} neurons representing n={n} features: {n/d:.1f}x overcomplete')
print(f'Sparsity: {sparse_codes.mean()*100:.1f}% of features active per sample')
print()
print('Interference matrix (dot products between feature directions):')
print(np.round(features_2d @ features_2d.T, 2))
print('Near-zero off-diagonal = nearly orthogonal => low interference')

---
## Section 2: The Sparse Autoencoder Architecture

A Sparse Autoencoder (SAE) adds two key ideas to a standard autoencoder:
1. **Overcomplete hidden layer**: hidden dim >> input dim (e.g., 4096 >> 512)
2. **L1 sparsity penalty**: forces most activations to be exactly zero

The result: a **dictionary** of features where each input is explained as a sparse sum of dictionary atoms.

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    Sparse Autoencoder with L1 sparsity penalty.
    
    Architecture:
        x (d) -> Encoder -> h (4d, sparse) -> Decoder -> x_hat (d)
        
    Key design choices:
    - Overcomplete: hidden_dim >> input_dim (overcomplete dictionary)
    - ReLU activation: non-negative activations, natural dead neurons
    - Decoder columns normalised: each column = unit-norm dictionary atom
    - Encoder bias: per-feature threshold (prevents trivial solutions)
    
    Training objective:
        L = ||x - x_hat||^2  +  lambda * ||h||_1
          reconstruction       sparsity penalty
    """
    def __init__(self, input_dim, hidden_dim, lambda_l1=0.01):
        super().__init__()
        self.input_dim  = input_dim
        self.hidden_dim = hidden_dim
        self.lambda_l1  = lambda_l1
        
        self.encoder = nn.Linear(input_dim, hidden_dim, bias=True)
        self.decoder = nn.Linear(hidden_dim, input_dim, bias=False)
        
        # Initialise decoder columns to unit norm
        with torch.no_grad():
            self.decoder.weight.data = nn.functional.normalize(
                self.decoder.weight.data, dim=0)
    
    def encode(self, x):
        """Encode: x -> sparse hidden activations h."""
        h = torch.relu(self.encoder(x))  # ReLU creates dead neurons
        return h
    
    def decode(self, h):
        """Decode: sparse h -> reconstruction x_hat."""
        return self.decoder(h)
    
    def forward(self, x):
        h     = self.encode(x)
        x_hat = self.decode(h)
        return x_hat, h
    
    def loss(self, x):
        """Compute total loss = reconstruction MSE + lambda * L1 sparsity."""
        x_hat, h = self.forward(x)
        recon_loss  = ((x - x_hat)**2).mean()            # MSE reconstruction
        sparse_loss = self.lambda_l1 * h.abs().mean()    # L1 sparsity penalty
        return recon_loss + sparse_loss, recon_loss.item(), sparse_loss.item()
    
    def normalise_decoder(self):
        """Keep decoder columns unit-norm after each gradient step."""
        with torch.no_grad():
            self.decoder.weight.data = nn.functional.normalize(
                self.decoder.weight.data, dim=0)


class TopKSparseAutoencoder(nn.Module):
    """
    TopK variant: keeps exactly k largest activations, zeros the rest.
    Avoids dead neuron problem and removes need to tune lambda.
    (Makhzani & Frey, 2013; Gao et al., 2024)
    """
    def __init__(self, input_dim, hidden_dim, k=10):
        super().__init__()
        self.k = k
        self.encoder = nn.Linear(input_dim, hidden_dim, bias=True)
        self.decoder = nn.Linear(hidden_dim, input_dim, bias=False)
        with torch.no_grad():
            self.decoder.weight.data = nn.functional.normalize(
                self.decoder.weight.data, dim=0)
    
    def encode(self, x):
        h = torch.relu(self.encoder(x))
        # Keep only top-k activations per sample
        topk_vals, topk_idx = torch.topk(h, self.k, dim=1)
        h_sparse = torch.zeros_like(h)
        h_sparse.scatter_(1, topk_idx, topk_vals)  # place top-k back
        return h_sparse
    
    def forward(self, x):
        h     = self.encode(x)
        x_hat = self.decoder(h)
        return x_hat, h

print('SAE classes defined.')
# Quick sanity check
model = SparseAutoencoder(input_dim=32, hidden_dim=128, lambda_l1=0.01)
x_test = torch.randn(16, 32)
x_hat, h = model(x_test)
print(f'Input:  {x_test.shape} -> Hidden: {h.shape} -> Output: {x_hat.shape}')
print(f'Sparsity: {(h == 0).float().mean()*100:.1f}% zeros (before training)')

---
## Figure 1 SAE Architecture & Sparsity

**Alt-text:** Two panels. Left: SAE architecture diagram showing input layer (4 nodes, purple), overcomplete hidden layer (8 nodes, with 3 amber 'ON' nodes and 5 grey 'off' nodes), and output/reconstruction layer (4 nodes, cyan). Arrows connect layers with high-alpha for active paths, low-alpha for inactive. Right: histogram comparing activation distributions dense AE (purple, near-Gaussian) vs SAE (amber, highly concentrated near zero with a sparse tail). 80% of SAE activations are exactly zero.

In [ ]:
# Generate Fig 1 inline
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG)
fig.suptitle('Figure 1 SAE Architecture & Sparsity Distribution',
             color=AMBR, fontsize=13, fontweight='bold', y=1.01)

ax2 = axes[1]; sa(ax2)
acts_dense = np.abs(np.random.randn(2000) * 0.8)
acts_sae   = np.concatenate([np.zeros(1600), np.abs(np.random.exponential(1.5, 400))])
ax2.hist(acts_dense, bins=40, color=PURP, alpha=0.7, density=True,
         label='Dense AE activations', edgecolor=BG, linewidth=0.5)
ax2.hist(acts_sae, bins=40, color=AMBR, alpha=0.7, density=True, hatch='///',
         label='SAE activations (80% = 0)', edgecolor=BG, linewidth=0.5)
ax2.set_xlabel('Activation magnitude')
ax2.set_ylabel('Density')
ax2.set_title('Activation Distribution\nSAE: 80% exactly zero = dictionary features', fontsize=10)
ax2.legend(fontsize=9, facecolor=GRID, edgecolor=MUTE, labelcolor=TEXT)

axes[0].axis('off'); axes[0].set_facecolor(CARD)
axes[0].text(0.5, 0.5, '[Architecture diagram\nsee sae_fig1_architecture.png]',
             ha='center', va='center', color=MUTE, fontsize=10,
             transform=axes[0].transAxes)

plt.tight_layout(pad=1.5)
plt.savefig('sae_fig1_architecture.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Figure 1 saved.')

---
## Section 3: L1 Penalty Why Not L2?

The choice of L1 (not L2) is crucial. L2 penalty shrinks activations toward zero but never reaches exactly zero. L1 has a **gradient discontinuity at zero** — the subgradient always points toward zero, creating exact sparsity.

In [ ]:
# Why L1 creates exact zeros but L2 does not
# For a scalar z with reconstruction loss r(z) and sparsity penalty:
# L2: d/dz [r(z) + lambda*z^2] = r'(z) + 2*lambda*z  -> zero only at z = -r'(z)/(2*lambda)
# L1: d/dz [r(z) + lambda*|z|] = r'(z) + lambda*sign(z) -> can push z to exactly 0

# The key geometric property: L1 ball has corners on axes, L2 ball is smooth
# Sparse solutions lie at the corners -> L1 encourages sparsity, L2 does not

# Demonstrate: optimise a simple quadratic + sparsity penalty
def optimise_with_penalty(target, penalty='l1', lambda_=0.5, steps=200):
    z = torch.tensor([2.0], requires_grad=True)
    opt = optim.SGD([z], lr=0.05)
    history = []
    for _ in range(steps):
        recon = (z - target)**2
        if penalty == 'l1':
            loss = recon + lambda_ * z.abs()
        else:
            loss = recon + lambda_ * z**2
        opt.zero_grad(); loss.backward(); opt.step()
        history.append(z.item())
    return history, z.item()

# With target=0.3 (small signal near zero):
hist_l1, z_l1 = optimise_with_penalty(torch.tensor(0.3), 'l1', lambda_=0.5)
hist_l2, z_l2 = optimise_with_penalty(torch.tensor(0.3), 'l2', lambda_=0.5)

print(f'Target: 0.3,  lambda: 0.5')
print(f'L1 final value: {z_l1:.6f}  <- pushed to EXACTLY zero')
print(f'L2 final value: {z_l2:.6f}  <- shrunk but NOT zero')
print()
print('L1 creates exact sparsity because:')
print('  gradient of |z| = sign(z) always pointing toward 0')
print('  once z=0, gradient is a subgradient in [-1,1], can stay at 0')
print('L2 creates soft sparsity because:')
print('  gradient of z^2 = 2z proportional to z, vanishes near 0')
print('  never fully reaches 0 unless reconstruction loss is exactly 0')

---
## Section 4: Training a SAE on Synthetic Data

We train a SAE on synthetic data generated from a known sparse dictionary. Ground truth: each sample is a sparse linear combination of 10 dictionary atoms in a 20-dimensional space. The SAE must recover these atoms in an 80-dimensional overcomplete hidden layer.

In [ ]:
# Generate synthetic sparse-dictionary data
D_TRUE = 10   # true number of dictionary atoms
N_FEAT = 20   # input dimension
N_DATA = 3000 # number of training samples
HIDDEN = 80   # SAE hidden dim (4x overcomplete)

torch.manual_seed(42); np.random.seed(42)

# True dictionary: D_TRUE atoms in N_FEAT dimensions
true_dict = np.random.randn(N_FEAT, D_TRUE)
true_dict = true_dict / np.linalg.norm(true_dict, axis=0, keepdims=True)  # unit-norm atoms

# Sparse codes: each sample uses ~2 atoms from D_TRUE
codes = np.zeros((N_DATA, D_TRUE))
for i in range(N_DATA):
    n_active = np.random.choice([1, 2, 3], p=[0.5, 0.4, 0.1])  # sparse!
    active = np.random.choice(D_TRUE, size=n_active, replace=False)
    codes[i, active] = np.random.exponential(1.0, size=n_active)

data = (codes @ true_dict.T).astype(np.float32)
data_t = torch.tensor(data)

print(f'Data shape: {data.shape}')
print(f'Average active features per sample: {(codes>0).sum(1).mean():.2f}')
print(f'Ground truth dictionary: {true_dict.shape}')
print(f'SAE hidden dim: {HIDDEN} ({HIDDEN//D_TRUE}x overcomplete vs ground truth)')

In [ ]:
# Train the SAE
model = SparseAutoencoder(input_dim=N_FEAT, hidden_dim=HIDDEN, lambda_l1=0.05)
opt   = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=500)

train_losses, recon_losses, sparse_losses = [], [], []

EPOCHS = 600
for ep in range(EPOCHS):
    model.train()
    idx   = torch.randint(0, N_DATA, (256,))
    total, recon, sparse = model.loss(data_t[idx])
    opt.zero_grad()
    total.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    model.normalise_decoder()  # keep dictionary atoms unit-norm
    scheduler.step()
    train_losses.append(total.item())
    recon_losses.append(recon)
    sparse_losses.append(sparse)
    if (ep+1) % 100 == 0:
        with torch.no_grad():
            _, h = model(data_t)
        dead = (h.numpy().max(0) == 0).mean() * 100
        print(f'Epoch {ep+1}/{EPOCHS}: total={total.item():.4f}, recon={recon:.4f}, '
              f'sparse={sparse:.4f}, dead={dead:.1f}%')

with torch.no_grad():
    _, h_all = model(data_t)
    h_np = h_all.numpy()

dead_pct = (h_np.max(0) == 0).mean() * 100
sparsity = (h_np == 0).mean() * 100
print(f'\nFinal: dead neurons={dead_pct:.1f}%, activation sparsity={sparsity:.1f}%')

---
## Figure 3 SAE Learns Sparse Dictionary Features

**Alt-text:** Three panels. Left: training curves showing total loss (amber), reconstruction MSE (purple dashed), and L1 sparsity term (rose dotted) over 600 epochs all three decrease and stabilise. Middle: neuron activation frequency bar chart sorted descending a fraction of neurons are highly active (amber), many are dead/rarely active (rose). Right: a 4x4 grid of mini bar charts showing the top 16 most-active SAE feature dictionary atoms (decoder columns), each showing the feature's profile across input dimensions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG)
fig.suptitle('Figure 3 SAE Training Curves & Activation Frequency',
             color=AMBR, fontsize=13, fontweight='bold', y=1.01)

ax = axes[0]; sa(ax)
eps = np.arange(len(train_losses))
ax.plot(eps, uniform_filter1d(train_losses,  10), color=AMBR, linewidth=2.5, label='Total loss')
ax.plot(eps, uniform_filter1d(recon_losses,  10), color=PURP, linewidth=2.0, linestyle='--', label='Reconstruction MSE')
ax.plot(eps, uniform_filter1d(sparse_losses, 10), color=ROSE, linewidth=2.0, linestyle=':', label='L1 sparsity term')
ax.set_xlabel('Training epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Curves\nTotal = Reconstruction + λ·L1', fontsize=10)
ax.legend(fontsize=9, facecolor=GRID, edgecolor=MUTE, labelcolor=TEXT)

ax2 = axes[1]; sa(ax2)
act_freq = (h_np > 0).mean(0)
sorted_freq = np.sort(act_freq)[::-1]
bar_colors = [AMBR if f > 0.01 else ROSE for f in sorted_freq]
ax2.bar(np.arange(len(sorted_freq)), sorted_freq, color=bar_colors, edgecolor=BG, linewidth=0.3, width=1.0)
ax2.axhline(0.01, color=LIME, linestyle='--', linewidth=1.5, label='1% threshold')
dead_pct_vis = (act_freq == 0).mean() * 100
ax2.set_xlabel('Neuron index (sorted by frequency)')
ax2.set_ylabel('Activation frequency')
ax2.set_title(f'Neuron Activation Frequency\n{dead_pct_vis:.0f}% dead neurons (exactly zero)', fontsize=10)
ax2.legend(fontsize=9, facecolor=GRID, edgecolor=MUTE, labelcolor=TEXT)
ax2.text(0.6, 0.85, f'{dead_pct_vis:.0f}% dead', transform=ax2.transAxes,
         color=ROSE, fontsize=11, fontweight='bold')

plt.tight_layout(pad=1.5)
plt.savefig('sae_fig3_features.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Figure 3 saved.')

---
## Section 5: Monosemanticity Analysis

The key measure of a SAE's success is **monosemanticity**: how many of its features correspond to a single, interpretable concept? We measure this via feature cosine similarity with the ground-truth dictionary atoms.

In [ ]:
# Measure recovery of ground-truth dictionary atoms
W_d = model.decoder.weight.detach().numpy()  # (input_dim, hidden_dim)

# Compute cosine similarity between each SAE atom and each true atom
# W_d columns are SAE atoms, true_dict columns are ground-truth atoms
similarity = np.abs(true_dict.T @ W_d)  # (D_TRUE, hidden_dim)

# For each true atom, find the best matching SAE atom
best_match_sim = similarity.max(axis=1)  # (D_TRUE,)
best_match_idx = similarity.argmax(axis=1)

print('Ground-truth atom recovery (cosine similarity to best SAE match):')
for i in range(D_TRUE):
    print(f'  True atom {i}: best SAE match = neuron {best_match_idx[i]:3d}, '
          f'similarity = {best_match_sim[i]:.3f}')
print()
print(f'Mean recovery similarity: {best_match_sim.mean():.3f}')
print(f'Atoms recovered with >0.8 similarity: {(best_match_sim>0.8).sum()}/{D_TRUE}')
print(f'Atoms recovered with >0.5 similarity: {(best_match_sim>0.5).sum()}/{D_TRUE}')

---
## Summary Table

| Model | Sparsity? | Overcomplete? | Monosemantic? | Primary Use |
|-------|-----------|---------------|---------------|-------------|
| Standard AE | No | No | Low | Compression |
| VAE | No | No | Partial | Generation |
| k-Sparse AE | YES | Optional | Medium | Dictionary learning |
| SAE (L1 penalty) | **YES** | **YES** | **HIGH** | LLM interpretability |
| TopK-SAE | **YES** | **YES** | **HIGH** | LLM interpretability |

## References

1. Olshausen, B.A. and Field, D.J. (1997) 'Sparse coding with an overcomplete basis set', *Vision Research*, 37(23), pp. 3311-3325. https://doi.org/10.1016/S0042-6989(97)00169-7
2. Bricken, T. et al. (2023) 'Towards Monosemanticity: Decomposing Language Models With Dictionary Learning'. https://transformer-circuits.pub/2023/monosemantic-features
3. Cunningham, H. et al. (2023) 'Sparse Autoencoders Find Highly Interpretable Features in Language Models'. https://arxiv.org/abs/2309.08600
4. Elhage, N. et al. (2022) 'Toy Models of Superposition'. https://transformer-circuits.pub/2022/toy_model
5. Makhzani, A. and Frey, B. (2013) 'k-Sparse Autoencoders'. https://arxiv.org/abs/1312.5663
6. Lee, H. et al. (2006) 'Efficient sparse coding algorithms'. https://arxiv.org/abs/cs/0608094